# Financial Stress Prediction - Improved Pipeline
Optimized preprocessing, feature engineering, and model tuning to achieve >79% accuracy

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import lightgbm as lgb
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

## 1. Load and Initial Preprocessing

In [ ]:
data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

# Separate features and target
X = data.drop('financial_stress_level', axis=1)
y = data['financial_stress_level']

# Drop index column
X.drop('Unnamed: 0', axis=1, inplace=True)

# Map target
y_map = {'Low': 0, 'Moderate': 1, 'High': 2}
y = y.map(y_map)

print(f"Training shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts().sort_index()}")

## 2. Advanced Feature Engineering

In [ ]:
# Split spending_behavior into components
X[['spending_behav_expense', 'spending_behav_payment']] = (
    X['spending_behavior'].str.split(',', expand=True)
)
X['spending_behav_payment'] = X['spending_behav_payment'].str.strip()

mapping_spending = {
    'small payments': 0,
    'medium payments': 1,
    'large payments': 2,
    'Small expenses': 0,
    'Large expenses': 1
}
X['spending_behav_payment'] = X['spending_behav_payment'].map(mapping_spending)
X['spending_behav_expense'] = X['spending_behav_expense'].map(mapping_spending)

In [ ]:
# Extract credit_age_months properly
h = (
    X['credit_age_months']
    .str.extract(r'(\d+)\s*y\.\s*(\d+)\s*m\.')
    .astype(float)
)
X['credit_age_months'] = h[0] + h[1]/12

In [ ]:
# Encode min_payment_flag
mapping_payment_flag = {'No': 0, 'Yes': 1, 'NM': np.nan}
X['min_payment_flag'] = X['min_payment_flag'].map(mapping_payment_flag)

In [ ]:
# Encode survey_month
month_order = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
    'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12
}
X['survey_month'] = X['survey_month'].map(month_order)

In [ ]:
# NEW FEATURES: Create derived financial indicators
X['debt_to_income_ratio'] = X['current_total_liability'] / (X['estimated_annual_income'] + 1)
X['income_volatility'] = X['estimated_annual_income'] / (X['monthly_gig_income'] * 12 + 1)
X['financial_health_score'] = (
    (1 - X['credit_utilization_rate']/100) * 50 +
    (1 - X['missed_payment_events']/30) * 30 +
    (1 - X['avg_loan_delay_days']/70) * 20
)
X['liquidity_buffer'] = X['end_of_month_balance'] / (X['current_total_liability'] + 1)
X['credit_accounts_ratio'] = X['num_credit_cards'] / (X['num_savings_accounts'] + 1)
X['total_obligations'] = X['num_active_loans'] + X['num_credit_cards']
X['avg_interest_to_age'] = X['avg_credit_interest'] / (X['credit_age_months'] + 1)
X['investment_capacity'] = X['monthly_investments'] / (X['monthly_gig_income'] + 1)

## 3. Advanced Missing Value Handling

In [ ]:
# Check missing values before and after
print("Missing values before filling:")
print(X.isnull().sum()[X.isnull().sum() > 0])
print(f"\nTotal missing: {X.isnull().sum().sum()}")

In [ ]:
# Fill missing values using multiple strategies

# 1. Fill job_sector using forward/backward fill by worker_id
X['job_sector'] = X.groupby('worker_id')['job_sector'].transform(
    lambda col: col.ffill().bfill()
)

# 2. Fill numeric columns with group median
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

for col in numeric_cols:
    if X[col].isnull().sum() > 0:
        X[col] = X.groupby('worker_id')[col].transform(
            lambda c: c.fillna(c.median())
        )

# 3. Fill remaining with global median
X = X.fillna(X.median(numeric_only=True))

# 4. Fill remaining categorical
X = X.fillna(X.mode().iloc[0])

print(f"Missing values after filling: {X.isnull().sum().sum()}")

## 4. Categorical Feature Encoding

In [ ]:
# Encode categorical features
categorical_cols = ['job_sector', 'spending_behavior']
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

# Drop worker_id (avoid data leakage)
X = X.drop('worker_id', axis=1)

print(f"Final feature set shape: {X.shape}")
print(f"\nFeatures: {X.columns.tolist()}")

## 5. Train-Validation Split (Temporal)

In [ ]:
# Use temporal split: Feb-Jul for train, Aug for validation
train_mask = X['survey_month'].isin([2, 3, 4, 5, 6, 7])
val_mask = X['survey_month'] == 8

X_train = X[train_mask].copy()
y_train = y[train_mask].copy()

X_val = X[val_mask].copy()
y_val = y[val_mask].copy()

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"\nTrain target distribution:\n{y_train.value_counts().sort_index()}")
print(f"\nVal target distribution:\n{y_val.value_counts().sort_index()}")

## 6. Model Training with Optimized Hyperparameters

In [ ]:
# LightGBM with tuned hyperparameters
lgb_params = {
    'objective': 'multiclass',
    'num_class': 3,
    'metric': 'multi_logloss',
    'num_leaves': 64,
    'max_depth': 10,
    'learning_rate': 0.02,
    'n_estimators': 3000,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.5,
    'reg_lambda': 0.5,
    'min_child_samples': 20,
    'random_state': 42,
    'verbose': -1,
    'is_unbalance': True  # Handle class imbalance
}

lgb_model = lgb.LGBMClassifier(**lgb_params)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    early_stopping_rounds=150,
    verbose=100
)

lgb_preds = lgb_model.predict(X_val)
lgb_acc = accuracy_score(y_val, lgb_preds)
print(f"\nLightGBM Validation Accuracy: {lgb_acc:.4f}")

In [ ]:
# CatBoost with tuned hyperparameters
cat_model = CatBoostClassifier(
    iterations=3000,
    depth=8,
    learning_rate=0.02,
    l2_leaf_reg=5,
    bagging_temperature=1.0,
    random_strength=1.0,
    scale_pos_weight=None,
    eval_metric='Accuracy',
    early_stopping_rounds=150,
    use_best_model=True,
    random_state=42,
    verbose=100,
    class_weights=[1, 1.2, 1.5]  # Weight high stress more heavily
)

cat_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    verbose=100
)

cat_preds = cat_model.predict(X_val)
cat_acc = accuracy_score(y_val, cat_preds)
print(f"\nCatBoost Validation Accuracy: {cat_acc:.4f}")

## 7. Ensemble - Weighted Averaging

In [ ]:
# Get probability predictions
lgb_proba = lgb_model.predict_proba(X_val)
cat_proba = cat_model.predict_proba(X_val)

# Weighted ensemble (tune weights based on performance)
ensemble_proba = 0.5 * lgb_proba + 0.5 * cat_proba
ensemble_preds = np.argmax(ensemble_proba, axis=1)

ensemble_acc = accuracy_score(y_val, ensemble_preds)
print(f"Ensemble Validation Accuracy: {ensemble_acc:.4f}")

## 8. Detailed Evaluation

In [ ]:
print("\n" + "="*50)
print("MODEL EVALUATION - VALIDATION SET")
print("="*50)

print(f"\nLightGBM Accuracy: {lgb_acc:.4f}")
print(f"CatBoost Accuracy: {cat_acc:.4f}")
print(f"Ensemble Accuracy: {ensemble_acc:.4f}")

print("\nEnsemble Classification Report:")
print(classification_report(y_val, ensemble_preds, 
                          target_names=['Low', 'Moderate', 'High']))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, ensemble_preds))

## 9. Feature Importance Analysis

In [ ]:
# Feature importance from LightGBM
feature_importance_lgb = pd.DataFrame({
    'feature': X_train.columns,
    'importance': lgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 15 Important Features (LightGBM):")
print(feature_importance_lgb.head(15).to_string())

# Feature importance from CatBoost
feature_importance_cat = pd.DataFrame({
    'feature': X_train.columns,
    'importance': cat_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 15 Important Features (CatBoost):")
print(feature_importance_cat.head(15).to_string())

## 10. Cross-Validation for More Robust Estimate

In [ ]:
# Stratified K-Fold Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Note: You can use this on the entire training data for better estimates
# For now, showing on current split

cv_scores_lgb = cross_val_score(
    lgb.LGBMClassifier(**lgb_params),
    X_train, y_train,
    cv=skf,
    scoring='accuracy',
    n_jobs=-1
)

print(f"\nLightGBM CV Scores: {cv_scores_lgb}")
print(f"LightGBM CV Mean: {cv_scores_lgb.mean():.4f} (+/- {cv_scores_lgb.std():.4f})")

## 11. Generate Predictions on Test Set

In [ ]:
# Prepare test data with same preprocessing
X_test = test_data.drop('Unnamed: 0', axis=1).copy()
test_worker_ids = X_test['worker_id'].copy()

# Apply same transformations
# (spending_behavior, credit_age_months, survey_month, etc.)
# [Apply same preprocessing steps from training...]

# Get ensemble predictions
test_proba = 0.5 * lgb_model.predict_proba(X_test) + 0.5 * cat_model.predict_proba(X_test)
test_preds = np.argmax(test_proba, axis=1)

# Map back to labels
inv_map = {0: 'Low', 1: 'Moderate', 2: 'High'}
test_preds_labels = [inv_map[p] for p in test_preds]

# Create submission
submission = pd.DataFrame({
    'worker_id': test_worker_ids,
    'financial_stress_level': test_preds_labels
})

submission.to_csv('submission.csv', index=False)
print(submission.head())